In [1]:
# Docling Deep Dive: Document Intelligence for RAG Pipelines
#
# Docling is NOT an OCR engine — it's a document CONVERTER.
# It uses OCR internally but adds:
#   - Layout detection (columns, headers, footers, figures)
#   - Reading order (which text block comes first?)
#   - Table structure (rows, columns, merged cells → DataFrame)
#   - Multi-format support (PDF, DOCX, PPTX, images, HTML)
#   - Export to Markdown, JSON, HTML for downstream LLM/RAG use

from ocr_gauntlet.utils import ensure_samples

ensure_samples()  # download if needed

PosixPath('/Users/slava/Projects/ocr-gauntlet/data/samples')

In [2]:
from docling.document_converter import DocumentConverter
from ocr_gauntlet.utils import SAMPLES_DIR

converter = DocumentConverter()
result = converter.convert(str(SAMPLES_DIR / "04_dense_form.png"))

print(result.document.export_to_markdown()[:1000])

## COMPETITIVE PRODUCT INTRODUCTION PROGRESS REPORT

TO:

MRS. K.A. SPARROW

FROM: R.G. Ryan

DATE: 12/10/96

MANUFACTURER: R. J. Reynolds

BRAND: Camel Menthol

TYPE OF PACKINGS: Full Flavor Box and Light Box

REPORTING PERIODS:

AUG

SEPT

- [ ] OCT

(Forward by the 10th of the following month.)

NOV

TEST MARKET GEOGRAPHY: All of Region 7.

PRICE POINT: FULL $11.89

PN $

(Indicate Distributor's Cost Per Carton)

SALES FORCE INVOLVEMENT: Merchandising the top tray of permanent counter displays and labeling

carton fixtures in the Camel section. Also placing metal signs and temporary counter displays.

DISTRIBUTORS - ACCEPTANCE/INTRO TERMS/INTRO DEALS:

Product is being introduced to all Direct Accounts in the Region. Acceptance is spotty at this time.

## DISTRIBUTOR INVOLVEMENT:

Assembly of promotional products and shipment to retail. Indianapolis Direct Accounts are reported to be receiving 81G1F product.

## CHAINS - ACCEPTANCE/MERCHANDISING ALLOWANCE

Chain acceptance has been 

In [3]:
import json

doc = result.document
print(f"Tables: {len(doc.tables)}, Figures: {len(doc.pictures)}")
print(json.dumps(doc.export_to_dict(), indent=2)[:2000])

Tables: 0, Figures: 0
{
  "schema_name": "DoclingDocument",
  "version": "1.9.0",
  "name": "04_dense_form",
  "origin": {
    "mimetype": "application/pdf",
    "binary_hash": 8315503575297341278,
    "filename": "04_dense_form.png"
  },
  "furniture": {
    "self_ref": "#/furniture",
    "children": [],
    "content_layer": "furniture",
    "name": "_root_",
    "label": "unspecified"
  },
  "body": {
    "self_ref": "#/body",
    "children": [
      {
        "$ref": "#/texts/0"
      },
      {
        "$ref": "#/texts/1"
      },
      {
        "$ref": "#/texts/2"
      },
      {
        "$ref": "#/texts/3"
      },
      {
        "$ref": "#/groups/0"
      },
      {
        "$ref": "#/texts/31"
      },
      {
        "$ref": "#/texts/32"
      },
      {
        "$ref": "#/texts/33"
      }
    ],
    "content_layer": "body",
    "name": "_root_",
    "label": "unspecified"
  },
  "groups": [
    {
      "self_ref": "#/groups/0",
      "parent": {
        "$ref": "#/body"
 

In [4]:
for i, table in enumerate(result.document.tables):
    print(f"\n📋 Table {i + 1}:")
    display(table.export_to_dataframe())

In [5]:
from docling.document_converter import DocumentConverter, ImageFormatOption
from docling.datamodel.pipeline_options import TesseractCliOcrOptions
from ocr_gauntlet.utils import SAMPLES_DIR

ocr_options = TesseractCliOcrOptions(force_full_page_ocr=True)
image_options = ImageFormatOption(ocr_options=ocr_options, do_table_structure=True)
converter = DocumentConverter(format_options={"image": image_options})

result = converter.convert(str(SAMPLES_DIR / "02_receipt.png"))
print(result.document.export_to_markdown()[:500])

TRAD KY TOAST CARTE ITEMS 1.00 SUBTTL PB-1 10% TOTAL CASH

28.182

28.182

2.818

31.000 31.000


In [6]:
import pytesseract
from PIL import Image
from ocr_gauntlet.utils import SAMPLES_DIR

img = Image.open(SAMPLES_DIR / "02_receipt.png")

raw = pytesseract.image_to_string(img)
structured = result.document.export_to_markdown()

print("=== Raw Tesseract ===")
print(raw[:500])
print("\n=== Docling + Tesseract ===")
print(structured[:500])

# Key: Docling preserves structure and reading order.
# For RAG pipelines, Docling output is dramatically more useful.

=== Raw Tesseract ===
TRAD KY TOAST CARTE 28. 182

ITEMS 1.00
SUBTTL 28. 182
PB-1 10% 2.818
TOTAL 31.000
CASH 31.000



=== Docling + Tesseract ===
TRAD KY TOAST CARTE ITEMS 1.00 SUBTTL PB-1 10% TOTAL CASH

28.182

28.182

2.818

31.000 31.000


In [7]:
# ┌────────────────────────────────┬─────────────────────────────┐
# │ Use Docling when...            │ Use raw VLM API when...     │
# ├────────────────────────────────┼─────────────────────────────┤
# │ You need markdown/JSON for RAG │ You need max OCR accuracy   │
# │ Mixed formats (PDF+DOCX+PPTX) │ Single document type        │
# │ Tables must become DataFrames  │ Plain text extraction only  │
# │ You need reading order         │ Speed matters most          │
# │ On-premise / air-gapped        │ Cloud APIs are fine         │
# │ Budget is zero (CPU-only)      │ Budget allows API costs     │
# └────────────────────────────────┴─────────────────────────────┘
#
# Best combo: Docling for structure + Gemini 3 Flash for hard pages.
print("Docling is infrastructure for RAG. VLMs are accuracy engines.")

Docling is infrastructure for RAG. VLMs are accuracy engines.
